<a href="https://colab.research.google.com/github/BhavanaVuyyuru/rag-document-search/blob/main/notebooks/rag_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Install required libraries
!pip install sentence-transformers -q
!pip install pandas numpy -q

print("✅ Libraries installed!")

✅ Libraries installed!


In [2]:
import pandas as pd
import os

os.makedirs("data", exist_ok=True)
os.makedirs("output", exist_ok=True)

# Sample clinical documents
documents = [
    {
        "doc_id": "D001",
        "category": "diabetes",
        "text": "Patient diagnosed with Type 2 diabetes. Prescribed metformin 500mg twice daily. Blood sugar levels at 180 mg/dL. Follow-up in 3 months."
    },
    {
        "doc_id": "D002",
        "category": "cardiology",
        "text": "Patient reports chest pain and shortness of breath. ECG shows irregular heartbeat. Referred to cardiologist. Prescribed aspirin 81mg daily."
    },
    {
        "doc_id": "D003",
        "category": "blood_pressure",
        "text": "Blood pressure reading 145/92 mmHg — classified as Stage 2 hypertension. Recommended low sodium diet, exercise, and lisinopril 10mg daily."
    },
    {
        "doc_id": "D004",
        "category": "diabetes",
        "text": "Follow-up visit for diabetes management. HbA1c improved from 9.2 to 7.8. Patient adhering to metformin. Added insulin glargine for better control."
    },
    {
        "doc_id": "D005",
        "category": "general",
        "text": "Annual wellness checkup completed. Cholesterol 195 mg/dL — within normal range. BMI 24.3. No abnormalities detected. Next checkup in 12 months."
    },
    {
        "doc_id": "D006",
        "category": "cardiology",
        "text": "Post cardiac surgery follow-up. Wound healing well. No signs of infection. Patient advised to continue blood thinners and cardiac rehab program."
    },
    {
        "doc_id": "D007",
        "category": "blood_pressure",
        "text": "Patient with history of hypertension presenting with severe headache and dizziness. BP 180/110 — hypertensive crisis. Emergency medication administered."
    },
    {
        "doc_id": "D008",
        "category": "general",
        "text": "Patient presents with fever 101.3F, cough, and fatigue for 5 days. Rapid flu test positive. Prescribed Tamiflu and rest for 7 days."
    }
]

df = pd.DataFrame(documents)
df.to_csv("data/clinical_documents.csv", index=False)

print(f"✅ Created {len(documents)} clinical documents")
print("\n📋 Document Categories:")
print(df["category"].value_counts())
print("\n📄 Sample document:")
print(df["text"][0])

✅ Created 8 clinical documents

📋 Document Categories:
category
diabetes          2
cardiology        2
blood_pressure    2
general           2
Name: count, dtype: int64

📄 Sample document:
Patient diagnosed with Type 2 diabetes. Prescribed metformin 500mg twice daily. Blood sugar levels at 180 mg/dL. Follow-up in 3 months.


In [3]:
from sentence_transformers import SentenceTransformer
import numpy as np

# Load free embedding model — no API key needed!
print("⏳ Loading embedding model...")
model = SentenceTransformer("all-MiniLM-L6-v2")
print("✅ Model loaded!")

# Load documents
df = pd.read_csv("data/clinical_documents.csv")
documents = df["text"].tolist()
doc_ids   = df["doc_id"].tolist()
categories = df["category"].tolist()

# Generate embeddings
print(f"\n⏳ Generating embeddings for {len(documents)} documents...")
embeddings = model.encode(documents, show_progress_bar=True)

print(f"\n✅ Embeddings generated!")
print(f"📐 Embedding shape: {embeddings.shape}")
print(f"   → {embeddings.shape[0]} documents")
print(f"   → {embeddings.shape[1]} dimensions per document")

# Save embeddings
np.save("output/embeddings.npy", embeddings)
print("\n💾 Embeddings saved to output/embeddings.npy")

⏳ Loading embedding model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Model loaded!

⏳ Generating embeddings for 8 documents...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


✅ Embeddings generated!
📐 Embedding shape: (8, 384)
   → 8 documents
   → 384 dimensions per document

💾 Embeddings saved to output/embeddings.npy


In [4]:
import numpy as np

def cosine_similarity(vec1, vec2):
    """Calculate similarity between two vectors"""
    dot_product  = np.dot(vec1, vec2)
    norm1        = np.linalg.norm(vec1)
    norm2        = np.linalg.norm(vec2)
    return dot_product / (norm1 * norm2)

def search(query, documents, doc_ids, categories, embeddings, top_k=3):
    """
    Search for most relevant documents given a query
    """
    # Embed the query
    query_embedding = model.encode([query])[0]

    # Calculate similarity with all documents
    scores = []
    for i, doc_embedding in enumerate(embeddings):
        score = cosine_similarity(query_embedding, doc_embedding)
        scores.append({
            "doc_id":     doc_ids[i],
            "category":   categories[i],
            "score":      round(float(score), 4),
            "text":       documents[i][:120] + "..."
        })

    # Sort by score
    scores = sorted(scores, key=lambda x: x["score"], reverse=True)
    return scores[:top_k]

print("✅ Search engine ready!")

✅ Search engine ready!


In [5]:
# Test queries
queries = [
    "diabetes medication and blood sugar management",
    "heart problems and chest pain",
    "high blood pressure treatment",
    "general health checkup",
    "post surgery recovery"
]

all_results = []

for query in queries:
    print(f"\n{'='*65}")
    print(f"🔍 Query: {query}")
    print(f"{'='*65}")

    results = search(
        query,
        documents,
        doc_ids,
        categories,
        embeddings,
        top_k=2
    )

    for rank, result in enumerate(results, 1):
        print(f"\n  #{rank} [{result['doc_id']}] Score: {result['score']}")
        print(f"  Category: {result['category']}")
        print(f"  {result['text']}")

        all_results.append({
            "query":    query,
            "rank":     rank,
            "doc_id":   result["doc_id"],
            "category": result["category"],
            "score":    result["score"],
            "snippet":  result["text"]
        })

print(f"\n\n✅ Searched {len(queries)} queries successfully!")


🔍 Query: diabetes medication and blood sugar management

  #1 [D001] Score: 0.5522
  Category: diabetes
  Patient diagnosed with Type 2 diabetes. Prescribed metformin 500mg twice daily. Blood sugar levels at 180 mg/dL. Follow-...

  #2 [D004] Score: 0.5249
  Category: diabetes
  Follow-up visit for diabetes management. HbA1c improved from 9.2 to 7.8. Patient adhering to metformin. Added insulin gl...

🔍 Query: heart problems and chest pain

  #1 [D002] Score: 0.6684
  Category: cardiology
  Patient reports chest pain and shortness of breath. ECG shows irregular heartbeat. Referred to cardiologist. Prescribed ...

  #2 [D006] Score: 0.3771
  Category: cardiology
  Post cardiac surgery follow-up. Wound healing well. No signs of infection. Patient advised to continue blood thinners an...

🔍 Query: high blood pressure treatment

  #1 [D003] Score: 0.5312
  Category: blood_pressure
  Blood pressure reading 145/92 mmHg — classified as Stage 2 hypertension. Recommended low sodium diet, exerc

In [6]:
# Save search results
results_df = pd.DataFrame(all_results)
results_df.to_csv("output/search_results.csv", index=False)

# Print summary
print("📊 RAG Pipeline Summary")
print("="*40)
print(f"✅ Documents indexed:    {len(documents)}")
print(f"✅ Embedding dimensions: {embeddings.shape[1]}")
print(f"✅ Queries processed:    {len(queries)}")
print(f"✅ Results generated:    {len(all_results)}")
print(f"\n📁 Files saved:")
print(f"   → output/embeddings.npy")
print(f"   → output/search_results.csv")
print(f"\n🎉 RAG Pipeline Complete!")
print("\n💡 How this connects to production:")
print("   In production → embeddings stored in Pinecone/Weaviate")
print("   Query results → fed into GPT-4 as context (RAG)")
print("   This enables → natural language search over any documents")

📊 RAG Pipeline Summary
✅ Documents indexed:    8
✅ Embedding dimensions: 384
✅ Queries processed:    5
✅ Results generated:    10

📁 Files saved:
   → output/embeddings.npy
   → output/search_results.csv

🎉 RAG Pipeline Complete!

💡 How this connects to production:
   In production → embeddings stored in Pinecone/Weaviate
   Query results → fed into GPT-4 as context (RAG)
   This enables → natural language search over any documents
